LDA code heavily adapted from the [following tutorial](https://www.geeksforgeeks.org/machine-learning/latent-dirichlet-allocation-and-topic-modelling/)

Install dependencies and load data

In [2]:
!pip install --upgrade gensim pyLDAvis spacy pandas scikit-learn

  Using cached gensim-4.4.0-cp311-cp311-win_amd64.whl (24.4 MB)
  Using cached pyLDAvis-3.4.1-py3-none-any.whl (2.6 MB)
  Using cached pandas-3.0.1-cp311-cp311-win_amd64.whl (9.9 MB)
  Attempting uninstall: pandas
    Found existing installation: pandas 2.3.3
    Uninstalling pandas-2.3.3:
      Successfully uninstalled pandas-2.3.3



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import spacy.cli
spacy.cli.download("en_core_web_md")

import pandas as pd
import warnings
import string
import spacy
import nltk
import gensim
import matplotlib.pyplot as plt
from gensim import corpora
from gensim.models import CoherenceModel
from gensim.models import Phrases
from gensim.models.phrases import Phraser
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis
from nltk.corpus import stopwords
import en_core_web_md
nltk.download('wordnet')
nltk.download('stopwords')
pyLDAvis.enable_notebook()
warnings.filterwarnings("ignore")

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\willi\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\willi\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [5]:
green_claims = pd.read_csv('../../output/analyzed/vagueness_analyzed.csv')

# Split by time period
groups_wayback = {
    "Wayback":     green_claims[green_claims['isWayback'] == True].copy(),
    "Non-Wayback": green_claims[green_claims['isWayback'] == False].copy(),
}
print("=== isWayback split ===")
for label, df in groups_wayback.items():
    print(f"  {label}: {len(df):,} claims")

# Split by organization
orgs = sorted(green_claims["Organization"].dropna().unique())
groups_org = {org: green_claims[green_claims['Organization'] == org].copy() for org in orgs}
print("\n=== Organization split ===")
for label, df in groups_org.items():
    print(f"  {label}: {len(df):,} claims")

# Split by organization and time period
groups_combined = {}
print("\n=== Organization + isWayback split ===")
for org in orgs:
    for wb_val, wb_label in [(True, "Wayback"), (False, "Non-Wayback")]:
        label = f"{org} | {wb_label}"
        sub = green_claims[
            (green_claims['Organization'] == org) &
            (green_claims['isWayback'] == wb_val)
        ].copy()
        if len(sub) > 0:
            groups_combined[label] = sub
            print(f"  {label}: {len(sub):,} claims")
        else:
            print(f"  Skipping empty group: {label}")

=== isWayback split ===
  Wayback: 4,022 claims
  Non-Wayback: 1,598 claims

=== Organization split ===
  Canadian Natural Resources: 534 claims
  Enbridge: 1,638 claims
  Imperial Oil: 449 claims
  Pembina Pipeline: 1,101 claims
  Shell Canada: 559 claims
  Suncor Energy: 1,339 claims

=== Organization + isWayback split ===
  Canadian Natural Resources | Wayback: 463 claims
  Canadian Natural Resources | Non-Wayback: 71 claims
  Enbridge | Wayback: 830 claims
  Enbridge | Non-Wayback: 808 claims
  Imperial Oil | Wayback: 403 claims
  Imperial Oil | Non-Wayback: 46 claims
  Pembina Pipeline | Wayback: 527 claims
  Pembina Pipeline | Non-Wayback: 574 claims
  Shell Canada | Wayback: 523 claims
  Shell Canada | Non-Wayback: 36 claims
  Suncor Energy | Wayback: 1,276 claims
  Suncor Energy | Non-Wayback: 63 claims


Preprocessing

In [6]:
# Remove common words and company names
stop_words = stopwords.words("english") + [
    "suncor",
    "suncor_energy",
    "shell",
    "shell_canada",
    "pembina",
    "pembina_pipeline",
    "enbridge",
    "canadian_natural",
    "canadian_natural_resources",
    "imperial",
    "imperial_oil"
]

def remove_stopwords(text):
    textArr = text.split(' ')
    rem_text = " ".join([i for i in textArr if i not in stop_words])
    return rem_text

In [7]:
def clean_text(text):
    delete_dict = {sp_char: '' for sp_char in string.punctuation}
    delete_dict[' '] = ' '
    table = str.maketrans(delete_dict)
    text1 = text.translate(table)
    textArr = text1.split()
    text2 = ' '.join([w for w in textArr if not w.isdigit() and len(w) > 3])
    return text2.lower()

In [8]:
nlp = en_core_web_md.load(disable=['parser', 'ner'])

def lemmatization(texts, allowed_postags=['NOUN', 'ADJ']):
    output = []
    for sent in texts:
        doc = nlp(sent)
        output.append(
            [token.lemma_.lower() for token in doc if token.pos_ in allowed_postags and token.lemma_.lower() not in stop_words])
    return output

In [9]:
def preprocess(df):
    df = df.copy()
    df['Sentence'] = df['Sentence'].apply(clean_text)
    df['# Words'] = df['Sentence'].apply(lambda x: len(str(x).split()))
    df['Sentence'] = df['Sentence'].apply(remove_stopwords)

    text_list = df['Sentence'].tolist()
    tokenized = lemmatization(text_list)

    bigram  = Phrases(tokenized, min_count=3, threshold=10)
    trigram = Phrases(bigram[tokenized], threshold=10)
    bigram_mod  = Phraser(bigram)
    trigram_mod = Phraser(trigram)
    tokenized = [trigram_mod[bigram_mod[doc]] for doc in tokenized]

    dictionary = corpora.Dictionary(tokenized)
    doc_term_matrix = [dictionary.doc2bow(sent) for sent in tokenized]

    return df, tokenized, dictionary, doc_term_matrix

preprocessed_wayback = {label: preprocess(df) for label, df in groups_wayback.items()}
preprocessed_org = {label: preprocess(df) for label, df in groups_org.items()}
preprocessed_combined = {label: preprocess(df) for label, df in groups_combined.items()}

LDA

In [10]:
def train_lda(dtm, dictionary, label):
    if not dtm:
        print(f"[{label}] Empty doc-term matrix, skipping.")
        return None, None
    model = gensim.models.ldamodel.LdaModel(
        corpus=dtm, id2word=dictionary,
        num_topics=5, random_state=100,
        chunksize=1000, passes=50, iterations=100
    )
    vis = gensimvis.prepare(model, dtm, dictionary)
    return model, vis

lda_wayback = {
    label: train_lda(v[3], v[2], label)
    for label, v in preprocessed_wayback.items()
}
lda_org = {
    label: train_lda(v[3], v[2], label)
    for label, v in preprocessed_org.items()
}
lda_combined = {
    label: train_lda(v[3], v[2], label)
    for label, v in preprocessed_combined.items()
}

Visualizations

In [11]:
lda_wayback["Wayback"][1]

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
3      0.036933  0.001962       1        1  30.941832
1     -0.032185 -0.162355       2        1  20.392951
4     -0.154187 -0.140125       3        1  19.992282
2     -0.141270  0.264519       4        1  15.649734
0      0.290709  0.035999       5        1  13.023201, topic_info=                 Term        Freq       Total Category  logprob  loglift
31             energy  763.000000  763.000000  Default  30.0000  30.0000
290             world  176.000000  176.000000  Default  29.0000  29.0000
50           emission  405.000000  405.000000  Default  28.0000  28.0000
202  renewable_energy  217.000000  217.000000  Default  27.0000  27.0000
166          business  300.000000  300.000000  Default  26.0000  26.0000
..                ...         ...         ...      ...      ...      ...
56             global   56.205612  298.736454   Topic5  -4.4583   0.3679
10            company   60.585355  607.847483   Topic5  -4.3833  -0.2674
16               sand   50.500777  247.925972   Topic5  -4.5653   0.4473
27            project   51.995398  644.146399   Topic5  -4.5362  -0.4783
41             future   40.806801  114.944535   Topic5  -4.7785   1.0028

[274 rows x 6 columns], token_table=      Topic      Freq            Term
term                                 
938       1  0.214380        addition
938       3  0.762241        addition
584       4  0.989449        alliance
1511      4  0.978431       ambitious
698       4  0.973406  ambitious_goal
...     ...       ...             ...
2295      5  0.979059      world_mark
63        1  0.279238            year
63        2  0.279238            year
63        3  0.285179            year
63        4  0.160413            year

[315 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[4, 2, 5, 3, 1])

In [12]:
lda_wayback["Non-Wayback"][1]

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
0      0.037279  0.097905       1        1  25.410376
2      0.129432 -0.031087       2        1  23.508699
1     -0.044237  0.148795       3        1  18.413450
3      0.076260 -0.130525       4        1  17.489097
4     -0.198734 -0.085089       5        1  15.178378, topic_info=                    Term        Freq       Total Category  logprob  loglift
61               project  295.000000  295.000000  Default  30.0000  30.0000
35                energy  342.000000  342.000000  Default  29.0000  29.0000
41             renewable   96.000000   96.000000  Default  28.0000  28.0000
122                 goal   58.000000   58.000000  Default  27.0000  27.0000
78                future   84.000000   84.000000  Default  26.0000  26.0000
..                   ...         ...         ...      ...      ...      ...
265           investment   14.357664   86.617896   Topic5  -5.1108   0.0881
57              facility   14.271333  107.492450   Topic5  -5.1168  -0.1339
374               system   12.015377   51.668173   Topic5  -5.2889   0.4266
115             strategy   12.016902   56.704947   Topic5  -5.2888   0.3338
58   greenhouse_emission   10.986310   35.358262   Topic5  -5.3784   0.7164

[348 rows x 6 columns], token_table=      Topic      Freq         Term
term                              
431       1  0.915322      ability
431       2  0.083211      ability
216       1  0.759532       access
216       4  0.210981       access
159       2  0.928396  acquisition
...     ...       ...          ...
92        3  0.126143         work
92        5  0.126143         work
69        2  0.432379         year
69        3  0.432379         year
69        4  0.131024         year

[534 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[1, 3, 2, 4, 5])

In [13]:
lda_org["Suncor Energy"][1]

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
4      0.185535 -0.055681       1        1  22.038092
0      0.005582  0.136367       2        1  20.884414
1     -0.197350 -0.033252       3        1  20.005748
3      0.014488  0.115020       4        1  18.606783
2     -0.008255 -0.162454       5        1  18.464963, topic_info=            Term        Freq       Total Category  logprob  loglift
56       project  189.000000  189.000000  Default  30.0000  30.0000
25        energy  198.000000  198.000000  Default  29.0000  29.0000
74    investment   77.000000   77.000000  Default  28.0000  28.0000
95      emission   93.000000   93.000000  Default  27.0000  27.0000
168  development   70.000000   70.000000  Default  26.0000  26.0000
..           ...         ...         ...      ...      ...      ...
193    portfolio   10.145118   20.416880   Topic5  -5.4192   0.9899
36    production   15.882719  112.414418   Topic5  -4.9709  -0.2677
150     canadian   11.155783   54.678835   Topic5  -5.3242   0.0998
18          sand   11.594661  116.755068   Topic5  -5.2856  -0.6202
78   significant   10.374926   40.763677   Topic5  -5.3968   0.3209

[331 rows x 6 columns], token_table=      Topic      Freq      Term
term                           
651       3  0.952742    access
1159      1  0.909589    action
1010      1  0.857562    active
424       2  0.478001  activity
424       3  0.286801  activity
...     ...       ...       ...
93        5  0.444557     world
33        1  0.182026      year
33        2  0.224032      year
33        3  0.056008      year
33        4  0.532076      year

[455 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[5, 1, 2, 4, 3])

In [14]:
lda_org["Pembina Pipeline"][1]

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
1     -0.119431 -0.104414       1        1  27.964469
2     -0.141800 -0.086683       2        1  21.620872
4     -0.012812  0.108543       3        1  19.807432
3      0.029481  0.202533       4        1  16.200326
0      0.244562 -0.119979       5        1  14.406902, topic_info=            Term        Freq       Total Category  logprob  loglift
26        energy  180.000000  180.000000  Default  30.0000  30.0000
158      company   58.000000   58.000000  Default  29.0000  29.0000
5        project  224.000000  224.000000  Default  28.0000  28.0000
23          year   77.000000   77.000000  Default  27.0000  27.0000
239     pipeline   79.000000   79.000000  Default  26.0000  26.0000
..           ...         ...         ...      ...      ...      ...
221     property    9.639653   23.566247   Topic5  -5.0280   1.0435
237     longterm    9.397918   36.170638   Topic5  -5.0534   0.5897
1    development    9.513588   58.832724   Topic5  -5.0411   0.1155
34      emission    9.420042   69.550153   Topic5  -5.0510  -0.0617
279         land    8.894329   15.194023   Topic5  -5.1084   1.4020

[303 rows x 6 columns], token_table=      Topic      Freq              Term
term                                   
877       1  0.877729            access
698       4  0.923201  access_education
841       2  0.539959          activity
841       4  0.449966          activity
478       5  0.939590        additional
...     ...       ...               ...
450       1  0.943761        worldclass
768       5  0.890133        worldscale
23        1  0.155550              year
23        2  0.168512              year
23        3  0.674049              year

[373 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[2, 3, 5, 4, 1])

In [15]:
lda_org["Imperial Oil"][1]

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
3     -0.005009 -0.012040       1        1  29.903015
1     -0.053435  0.147725       2        1  23.666726
2     -0.132571 -0.074109       3        1  20.313022
4      0.157358  0.000655       4        1  15.489512
0      0.033656 -0.062231       5        1  10.627725, topic_info=                              Term       Freq      Total Category  logprob  \
5                          percent  29.000000  29.000000  Default  30.0000   
2              greenhouse_emission  48.000000  48.000000  Default  29.0000   
40                        hydrogen  21.000000  21.000000  Default  28.0000   
13   greenhouse_emission_intensity  21.000000  21.000000  Default  27.0000   
186                           sand  29.000000  29.000000  Default  26.0000   
..                             ...        ...        ...      ...      ...   
463                         barrel   3.131388  15.105108   Topic5  -5.1314   
148                         corson   3.129899  16.672735   Topic5  -5.1319   
30                     significant   3.131379  18.838208   Topic5  -5.1314   
12                      efficiency   2.789458  10.655880   Topic5  -5.2470   
462                         annual   2.821660  12.166750   Topic5  -5.2356   

     loglift  
5    30.0000  
2    29.0000  
40   28.0000  
13   27.0000  
186  26.0000  
..       ...  
463   0.6681  
148   0.5689  
30    0.4473  
12    0.9014  
462   0.7803  

[344 rows x 6 columns], token_table=      Topic      Freq     Term
term                          
85        1  0.183193  ability
85        2  0.595377  ability
85        3  0.183193  ability
85        4  0.045798  ability
342       2  0.844832     able
...     ...       ...      ...
390       4  0.201307    world
390       5  0.100653    world
290       1  0.500387     year
290       2  0.050039     year
290       3  0.450348     year

[539 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[4, 2, 3, 5, 1])

In [16]:
lda_org["Enbridge"][1]

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
4      0.185689 -0.023033       1        1  26.391893
2      0.023858 -0.059876       2        1  21.887808
0      0.104620  0.006784       3        1  20.462010
1     -0.109043  0.214513       4        1  16.227749
3     -0.205124 -0.138389       5        1  15.030539, topic_info=          Term        Freq       Total Category  logprob  loglift
7       energy  676.000000  676.000000  Default  30.0000  30.0000
94     project  325.000000  325.000000  Default  29.0000  29.0000
40   renewable  267.000000  267.000000  Default  28.0000  28.0000
193       goal  104.000000  104.000000  Default  27.0000  27.0000
15    emission  159.000000  159.000000  Default  26.0000  26.0000
..         ...         ...         ...      ...      ...      ...
34      future   15.279579   92.232489   Topic5  -4.9878   0.0973
167   facility   14.360742   80.410890   Topic5  -5.0499   0.1724
90     company   14.611752  126.016625   Topic5  -5.0325  -0.2595
210       year   13.512074   73.940184   Topic5  -5.1108   0.1954
121   business   14.550603  132.693310   Topic5  -5.0367  -0.3153

[329 rows x 6 columns], token_table=      Topic      Freq         Term
term                              
81        2  0.262308  acquisition
81        4  0.682000  acquisition
448       4  0.833835  action_plan
418       1  0.248665     addition
418       2  0.683828     addition
...     ...       ...          ...
210       1  0.175818         year
210       2  0.473356         year
210       3  0.054098         year
210       4  0.094671         year
210       5  0.189342         year

[481 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[5, 3, 1, 2, 4])

In [17]:
lda_org["Canadian Natural Resources"][1]

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
3      0.037824 -0.000233       1        1  28.212092
1      0.067329 -0.122883       2        1  22.264041
4     -0.158572 -0.069964       3        1  17.948811
0     -0.077592  0.134741       4        1  16.142745
2      0.131011  0.058337       5        1  15.432310, topic_info=                 Term        Freq       Total Category  logprob  loglift
144          industry   32.000000   32.000000  Default  30.0000  30.0000
11          operation   30.000000   30.000000  Default  29.0000  29.0000
26   canadian_natural   80.000000   80.000000  Default  28.0000  28.0000
33   methane_emission   20.000000   20.000000  Default  27.0000  27.0000
6             company  105.000000  105.000000  Default  26.0000  26.0000
..                ...         ...         ...      ...      ...      ...
76         production    7.860157   67.245333   Topic5  -4.5929  -0.2778
15        significant    6.356373   28.366193   Topic5  -4.8052   0.3730
152              year    6.752675   42.481100   Topic5  -4.7447   0.0296
144          industry    5.863248   32.019679   Topic5  -4.8860   0.1711
58              crude    4.800601   33.562961   Topic5  -5.0859  -0.0760

[330 rows x 6 columns], token_table=      Topic      Freq         Term
term                              
395       1  0.261659  abandonment
395       2  0.654148  abandonment
396       1  0.252675      ability
396       2  0.673800      ability
396       3  0.084225      ability
...     ...       ...          ...
152       1  0.282479         year
152       2  0.117699         year
152       3  0.400178         year
152       4  0.047080         year
152       5  0.164779         year

[457 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[4, 2, 5, 1, 3])

In [18]:
lda_org["Shell Canada"][1]

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
4      0.022148 -0.046614       1        1  28.048321
1     -0.032236 -0.114061       2        1  20.114417
0     -0.040691  0.095646       3        1  17.604351
3     -0.090311  0.034259       4        1  17.179158
2      0.141090  0.030770       5        1  17.053752, topic_info=         Term        Freq       Total Category  logprob  loglift
4      energy  117.000000  117.000000  Default  30.0000  30.0000
245     quest   30.000000   30.000000  Default  29.0000  29.0000
17    project   60.000000   60.000000  Default  28.0000  28.0000
252     tonne   10.000000   10.000000  Default  27.0000  27.0000
88   canadian   35.000000   35.000000  Default  26.0000  26.0000
..        ...         ...         ...      ...      ...      ...
235      work    4.018752   10.242490   Topic5  -5.3337   0.8332
63     school    3.721165   10.324023   Topic5  -5.4107   0.7484
54     design    3.488165    9.501627   Topic5  -5.4753   0.7667
118  business    3.757213   24.704651   Topic5  -5.4010  -0.1145
18      water    3.625609   21.937906   Topic5  -5.4367  -0.0314

[348 rows x 6 columns], token_table=      Topic      Freq          Term
term                               
1182      1  0.822798    accelerate
631       1  0.736169      addition
451       2  0.845077      advanced
877       2  0.950305     affiliate
330       1  0.116682  announcement
...     ...       ...           ...
52        2  0.198892          year
52        3  0.198892          year
52        4  0.159114          year
52        5  0.278449          year
962       2  0.845063  zeroemission

[559 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[5, 2, 1, 4, 3])